<a href="https://colab.research.google.com/github/JAHNAVI-BOOPATHY/deeplearning/blob/main/machine_translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ==========================================
# Machine Translation using Seq2Seq LSTM
# Google Colab Compatible
# ==========================================

import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ==========================================
# Dataset
# ==========================================

english = [
    "hello",
    "how are you",
    "good morning",
    "thank you",
    "i love you",
    "what is your name",
    "good night",
    "see you",
    "welcome",
    "yes"
]

french = [
    "bonjour",
    "comment allez vous",
    "bonjour",
    "merci",
    "je t aime",
    "quel est votre nom",
    "bonne nuit",
    "a bientot",
    "bienvenue",
    "oui"
]

decoder_input_texts = ["start " + text for text in french]
decoder_target_texts = [text + " end" for text in french]

# ==========================================
# Tokenization
# ==========================================

eng_tokenizer = Tokenizer(filters='')
eng_tokenizer.fit_on_texts(english)

fra_tokenizer = Tokenizer(filters='')
fra_tokenizer.fit_on_texts(decoder_input_texts + decoder_target_texts)

encoder_sequences = eng_tokenizer.texts_to_sequences(english)
decoder_input_sequences = fra_tokenizer.texts_to_sequences(decoder_input_texts)
decoder_target_sequences = fra_tokenizer.texts_to_sequences(decoder_target_texts)

max_encoder_len = max(len(seq) for seq in encoder_sequences)
max_decoder_len = max(len(seq) for seq in decoder_input_sequences)

encoder_input_data = pad_sequences(
    encoder_sequences,
    maxlen=max_encoder_len,
    padding="post"
)

decoder_input_data = pad_sequences(
    decoder_input_sequences,
    maxlen=max_decoder_len,
    padding="post"
)

decoder_target_data = pad_sequences(
    decoder_target_sequences,
    maxlen=max_decoder_len,
    padding="post"
)

decoder_target_data = np.expand_dims(decoder_target_data, -1)

num_encoder_tokens = len(eng_tokenizer.word_index) + 1
num_decoder_tokens = len(fra_tokenizer.word_index) + 1

# ==========================================
# Build Model
# ==========================================

latent_dim = 128

# Encoder
encoder_inputs = Input(shape=(None,))
encoder_embedding = Embedding(num_encoder_tokens, latent_dim)
encoder_emb = encoder_embedding(encoder_inputs)

encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_emb)

encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None,))
decoder_embedding = Embedding(num_decoder_tokens, latent_dim)
decoder_emb = decoder_embedding(decoder_inputs)

decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_emb,
    initial_state=encoder_states
)

decoder_dense = Dense(
    num_decoder_tokens,
    activation="softmax"
)

decoder_outputs = decoder_dense(decoder_outputs)

model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ==========================================
# Training
# ==========================================

model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=2,
    epochs=300,
    verbose=1
)

print("\nTraining Completed Successfully!")

# ==========================================
# Encoder Inference Model
# ==========================================

encoder_model = Model(
    encoder_inputs,
    encoder_states
)

# ==========================================
# Decoder Inference Model
# ==========================================

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))

decoder_states_inputs = [
    decoder_state_input_h,
    decoder_state_input_c
]

decoder_single_input = Input(shape=(1,))

decoder_single_emb = decoder_embedding(decoder_single_input)

decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_single_emb,
    initial_state=decoder_states_inputs
)

decoder_states = [state_h, state_c]

decoder_outputs = decoder_dense(decoder_outputs)

decoder_model = Model(
    [decoder_single_input] + decoder_states_inputs,
    [decoder_outputs] + decoder_states
)

# ==========================================
# Dictionaries
# ==========================================

reverse_target_index = {
    i: word
    for word, i in fra_tokenizer.word_index.items()
}

target_index = fra_tokenizer.word_index

# ==========================================
# Translation Function
# ==========================================

def translate(sentence):

    seq = eng_tokenizer.texts_to_sequences([sentence])

    seq = pad_sequences(
        seq,
        maxlen=max_encoder_len,
        padding="post"
    )

    states = encoder_model.predict(seq, verbose=0)

    target_seq = np.array([[target_index["start"]]])

    translated = ""

    while True:

        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states,
            verbose=0
        )

        sampled_token = np.argmax(output_tokens[0, -1, :])

        sampled_word = reverse_target_index.get(sampled_token, "")

        if sampled_word == "end" or sampled_word == "":
            break

        translated += sampled_word + " "

        target_seq = np.array([[sampled_token]])

        states = [h, c]

    return translated.strip()

# ==========================================
# Testing
# ==========================================

tests = [
    "hello",
    "thank you",
    "good night",
    "welcome",
    "yes",
    "i love you",
    "how are you",
    "see you"
]

print("\n========== Translation ==========\n")

for sentence in tests:

    print("English :", sentence)

    print("French  :", translate(sentence))

    print("-" * 35)

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_9       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, None, 128) │      2,304 │ input_layer_8[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, None, 128) │      2,688 │ input_layer_9[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ [(None, 128),     │    131,584 │ embedding_5[0][0] │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ [(None, None,     │    131,584 │ embedding_6[0][0… │
│                     │ 128), (None,      │            │ lstm_4[0][1],     │
│                     │ 128), (None,      │            │ lstm_4[0][2]      │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, None, 21)  │      2,709 │ lstm_5[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 270,869 (1.03 MB)

 Trainable params: 270,869 (1.03 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.3000 - loss: 3.0162  
Epoch 2/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4200 - loss: 2.8820
Epoch 3/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4200 - loss: 2.6128
Epoch 4/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4200 - loss: 1.9641
Epoch 5/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4200 - loss: 1.8829
Epoch 6/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4200 - loss: 1.6856
Epoch 7/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4800 - loss: 1.6077
Epoch 8/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6200 - loss: 1.5448
Epoch 9/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6000 - loss: 1.4649
Epoch 10/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5800 - loss: 1.4321
Epoch 11/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6200 - loss: 1.3724
Epoch 12/300
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6200 - 

French  : bonjour
-----------------------------------
English : thank you
French  : merci
-----------------------------------
English : good night
French  : bonne nuit
-----------------------------------
English : welcome
French  : bienvenue
-----------------------------------
English : yes
French  : oui
-----------------------------------
English : i love you
French  : je t aime
-----------------------------------
English : how are you
French  : comment allez vous
-----------------------------------
English : see you
French  : a bientot
-----------------------------------
